# 01 — source data, study design and audit

this notebook covers descriptive work for nacc path c. hypothesis tests are not part of phase 1.

## 1. where the data came from
nacc collects information through alzheimer's disease research centers. this zip contains an already-matched subset. we do not know the original data release or how visits and scans were chosen. the results describe this subset only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
                                                                                
ROOT = Path.cwd()
if not (ROOT / 'README.md').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'raw'
assert DATA.exists(), 'Place the three original CSVs in data/raw/; read data/README.md'
u = pd.read_csv(DATA / 'udslong_imputed.csv', low_memory=False)
m = pd.read_csv(DATA / 'mriscanclean.csv', low_memory=False)
print('UDS:', u.shape, 'MRI:', m.shape)


## 2. counting participants
one row per person in this prepared file does not mean each person attended only once. the original visits are not included.

In [ ]:
for label, df in [('UDS',u),('MRI',m)]:
    print(label, 'rows =',len(df), 'unique participants =',df.NACCID.nunique(),
          'duplicated IDs =',df.NACCID.duplicated().sum())
print('visit/date fields present:', sorted(set(u.columns) & {'NACCVNUM','VISITMO','VISITDAY','VISITYR','NACCFDYS'}))

## 3. checking missing and unusual values
missing-value codes depend on the variable. i check a small set of relevant columns rather than deleting every 8 or 9 as if it meant missing.

In [ ]:
for c in ['CDRSUM','NACCUDSD','SEX','EDUC','MOCATOTS','NACCBMI']:
    print(c, 'raw NaNs:', u[c].isna().sum(), 'unique:',u[c].nunique(),
          'most common:', u[c].value_counts().head(7).to_dict())
print('EDUC=99:', (u.EDUC==99).sum())
print('MoCA outside 0-30:', (~u.MOCATOTS.between(0,30)).sum())
print('MoCA fractional:', ((u.MOCATOTS%1)!=0).sum())
print('MRI nonpositive gray volumes:', (m.filter(regex='_GVOL$')<=0).sum().sum())

## 4. saved audit tables
the analysis script creates the variable list and quality checks. only summary information is exported. participant ids stay in the source data.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,str(ROOT/'src'/'phase1_analysis.py')],check=True)
pd.read_csv(ROOT/'outputs'/'tables'/'quality_audit.csv').head(20)

we cannot finish the visit-history or mri-selection comparison from this zip alone. the missing parent cohort and date information are listed in the report.